In [4]:
import os
import re
import sys
import argparse
import subprocess
import matplotlib
import time
import pandas as pd
import numpy as np
from itertools import groupby
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
import pickle
import csv
import xgboost as xgb
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from collections import defaultdict
import operator
import joblib
from sklearn.ensemble import BaggingClassifier
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn import preprocessing
import pybedtools as pbt
import pyBigWig as pbw
import glob
import time
from sklearn.preprocessing import StandardScaler
from datetime import date

In [5]:
#all cooking things
os.chdir("/home/pavelkrav/Release/") # заходим в корень

root = os.getcwd() # просто корень, где будем работать
basicdir = os.path.abspath('TF-ML/') # директория для общих файлов
outputdir = os.path.abspath('TF-ML/outputdir') # директория для подготовки данных
train_dir = os.path.abspath('TF-ML/train/') 
test_dir = os.path.abspath('TF-ML/test/') 

if not os.path.exists(basicdir): # если нет result, то создаем
    os.makedirs(basicdir)
if not os.path.exists(outputdir): # если нет outputdir, то создаем
    os.makedirs(outputdir)
if not os.path.exists(train_dir): # если нет outputdir, то создаем
    os.makedirs(train_dir)
if not os.path.exists(test_dir): # если нет outputdir, то создаем
    os.makedirs(test_dir)

pwmdir_mono = os.path.abspath('/home/pavelkrav/hocomoco11/models/pwm/mono/all') # здесь лежат все PWM каждая в своей директории
pwmdir_di = os.path.abspath('/home/pavelkrav/hocomoco11/models/pwm/di/all')


files_pwm_HUMAN_mono = []
files_pwm_MOUSE_mono = []

for d in os.listdir(pwmdir_mono): # для всех директорий с PWM 
    #print(d)
    d = pwmdir_mono + "/" + d # готовим путь
    if os.path.isdir(d): # если директория по пути существует
        os.chdir(d) # заходим в нее
        #print(d)
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm']) # забираем PWM
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_mono.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.pwm']) # забираем PWM
    os.chdir(pwmdir_mono) # выходим

print(len(files_pwm_HUMAN_mono))
print(len(files_pwm_MOUSE_mono))


files_pwm_HUMAN_di = []
files_pwm_MOUSE_di = []

for d in os.listdir(pwmdir_di): # для всех директорий с PWM 
    #print(d)
    d = pwmdir_di + "/" + d # готовим путь
    if os.path.isdir(d): # если директория по пути существует
        os.chdir(d) # заходим в нее
        #print(d)
        if d.split("_")[-1] == "HUMAN":
            files_pwm_HUMAN_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm']) # забираем PWM
        if d.split("_")[-1] == "MOUSE":
            files_pwm_MOUSE_di.append([f for f in os.listdir(os.getcwd()) if os.path.splitext(f)[1] == '.dpwm']) # забираем PWM
    os.chdir(pwmdir_di) # выходим

print(len(files_pwm_HUMAN_di))
print(len(files_pwm_MOUSE_di))


def remove_duplicates(l):
    return list(set(l))

print("We have {N} TF before filtration of mono".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_mono + files_pwm_HUMAN_mono]))))
print("We have {N} TF before filtration of di".format(N=len(remove_duplicates([x[0].split("~")[0].split("_")[0] for x in files_pwm_MOUSE_di + files_pwm_HUMAN_di]))))


796
502
270
242
We have 835 TF before filtration of mono
We have 369 TF before filtration of di


In [6]:
size_of_set = 10000
outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_control.fasta"

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"


In [7]:
#size_of_set = 1000
headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_control.csv"


In [8]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [9]:
trimmed = False
size = 20
begin_trim = size
end_trim = size
if trimmed == True:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
else:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"


In [10]:

with open(headerStr_names_outfile_HUMAN_train, "r") as output:
    headerStr_names_HUMAN_train = output.read().split('\n')
    print(headerStr_names_HUMAN_train[:10])

with open(headerStr_names_outfile_MOUSE_train, "r") as output:
    headerStr_names_MOUSE_train = output.read().split('\n')
    print(headerStr_names_MOUSE_train[:10])

with open(headerStr_names_outfile_HUMAN_control, "r") as output:
    headerStr_names_HUMAN_control = output.read().split('\n')
    print(headerStr_names_HUMAN_control[:10])

with open(headerStr_names_outfile_MOUSE_control, "r") as output:
    headerStr_names_MOUSE_control = output.read().split('\n')
    print(headerStr_names_MOUSE_control[:10])    
    

['ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_0_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_1_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_2_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_3_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_4_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_5_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_6_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_7_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_8_train', 'ANDR_HUMAN_PEAKS000001_macs_train_globalInd_0_localInd_9_train']
['ANDR_MOUSE_PEAKS000001_macs_train_globalInd_0_localInd_0_train', 'ANDR_MOUSE_PEAKS000001_macs_train_globalInd_0_localInd_1_train', 'ANDR_MOUSE_PEAKS000001_macs_train_globalInd_0_localInd_2_train', 'ANDR_MOUSE_PEAKS000001_macs_train_globalInd_0_localInd_3_train', 'ANDR_MOUSE_PEAKS000001_macs_train_globalInd_0_localInd_4_train', 'ANDR_MO

In [13]:
family_annotation = pd.read_csv("/home/pavelkrav/TF-ML_last_v_22/core_annotation_HUMAN_mono.csv", sep='\t')
#print(family_annotation.head())


def find_TF_family_dict(family_annotation):
    TF_family_dict = defaultdict(None)
    for i, j in zip(family_annotation["Model"], family_annotation["TF family"]):
        #print(i,j)
        if type(j) == str:
            #print(i.split('_')[0], j.split('{')[1][:3])
            TF_family_dict[i.split('_')[0]] = j.split('{')[1][:3]
    return TF_family_dict

TF_family_dict = find_TF_family_dict(family_annotation)




In [14]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [15]:
selected_HUMAN = ['ANDR_HUMAN',
 'AP2A_HUMAN',
 'BATF_HUMAN',
 'CEBPB_HUMAN',
 'COE1_HUMAN',
 'CTCF_HUMAN',
 'E2F4_HUMAN',
 'ERG_HUMAN',
 'ESR1_HUMAN',
 'FLI1_HUMAN',
 'GATA1_HUMAN',
 'GATA2_HUMAN',
 'GATA3_HUMAN',
 'GCR_HUMAN',
 'HNF4A_HUMAN',
 'IRF1_HUMAN',
 'IRF4_HUMAN',
 'JUND_HUMAN',
 'MAFK_HUMAN',
 'MAX_HUMAN',
 'MYC_HUMAN',
 'MYOD1_HUMAN',
 'P53_HUMAN',
 'PPARG_HUMAN',
 'PRGR_HUMAN',
 'REST_HUMAN',
 'RUNX1_HUMAN',
 'RXRA_HUMAN',
 'SOX2_HUMAN',
 'SPI1_HUMAN',
 'SRF_HUMAN',
 'STA5A_HUMAN',
 'STAT1_HUMAN',
 'STAT3_HUMAN',
 'TAL1_HUMAN',
 'TF65_HUMAN',
 'TFE2_HUMAN',
 'USF2_HUMAN']

selected_MOUSE = ['ANDR_MOUSE',
 'AP2A_MOUSE',
 'BATF_MOUSE',
 'CEBPB_MOUSE',
 'COE1_MOUSE',
 'CTCF_MOUSE',
 'E2F4_MOUSE',
 'ERG_MOUSE',
 'ESR1_MOUSE',
 'FLI1_MOUSE',
 'GATA1_MOUSE',
 'GATA2_MOUSE',
 'GATA3_MOUSE',
 'GCR_MOUSE',
 'HNF4A_MOUSE',
 'IRF1_MOUSE',
 'IRF4_MOUSE',
 'JUND_MOUSE',
 'MAFK_MOUSE',
 'MAX_MOUSE',
 'MYC_MOUSE',
 'MYOD1_MOUSE',
 'P53_MOUSE',
 'PPARG_MOUSE',
 'PRGR_MOUSE',
 'REST_MOUSE',
 'RUNX1_MOUSE',
 'RXRA_MOUSE',
 'SOX2_MOUSE',
 'SPI1_MOUSE',
 'SRF_MOUSE',
 'STA5A_MOUSE',
 'STAT1_MOUSE',
 'STAT3_MOUSE',
 'TAL1_MOUSE',
 'TF65_MOUSE',
 'TFE2_MOUSE',
 'USF2_MOUSE']

In [16]:
selected_HUMAN

['ANDR_HUMAN',
 'AP2A_HUMAN',
 'BATF_HUMAN',
 'CEBPB_HUMAN',
 'COE1_HUMAN',
 'CTCF_HUMAN',
 'E2F4_HUMAN',
 'ERG_HUMAN',
 'ESR1_HUMAN',
 'FLI1_HUMAN',
 'GATA1_HUMAN',
 'GATA2_HUMAN',
 'GATA3_HUMAN',
 'GCR_HUMAN',
 'HNF4A_HUMAN',
 'IRF1_HUMAN',
 'IRF4_HUMAN',
 'JUND_HUMAN',
 'MAFK_HUMAN',
 'MAX_HUMAN',
 'MYC_HUMAN',
 'MYOD1_HUMAN',
 'P53_HUMAN',
 'PPARG_HUMAN',
 'PRGR_HUMAN',
 'REST_HUMAN',
 'RUNX1_HUMAN',
 'RXRA_HUMAN',
 'SOX2_HUMAN',
 'SPI1_HUMAN',
 'SRF_HUMAN',
 'STA5A_HUMAN',
 'STAT1_HUMAN',
 'STAT3_HUMAN',
 'TAL1_HUMAN',
 'TF65_HUMAN',
 'TFE2_HUMAN',
 'USF2_HUMAN']

In [17]:
selected_MOUSE[6:]

['E2F4_MOUSE',
 'ERG_MOUSE',
 'ESR1_MOUSE',
 'FLI1_MOUSE',
 'GATA1_MOUSE',
 'GATA2_MOUSE',
 'GATA3_MOUSE',
 'GCR_MOUSE',
 'HNF4A_MOUSE',
 'IRF1_MOUSE',
 'IRF4_MOUSE',
 'JUND_MOUSE',
 'MAFK_MOUSE',
 'MAX_MOUSE',
 'MYC_MOUSE',
 'MYOD1_MOUSE',
 'P53_MOUSE',
 'PPARG_MOUSE',
 'PRGR_MOUSE',
 'REST_MOUSE',
 'RUNX1_MOUSE',
 'RXRA_MOUSE',
 'SOX2_MOUSE',
 'SPI1_MOUSE',
 'SRF_MOUSE',
 'STA5A_MOUSE',
 'STAT1_MOUSE',
 'STAT3_MOUSE',
 'TAL1_MOUSE',
 'TF65_MOUSE',
 'TFE2_MOUSE',
 'USF2_MOUSE']

In [ ]:
# Сканируем mfa с помощью матриц. Запускать 1 раз!

for seq_org in ["HUMAN", "MOUSE"]:
    for pwm_org in ["HUMAN"]:

        # константы HUMAN_HUMAN mono
        N_features = 1000
        seed(666)
        model = "XGBClassifier"
        organism = "HUMAN_MOUSE"

        if pwm_org == "HUMAN":
            sel_TF = selected_HUMAN
        else:
            sel_TF = selected_MOUSE
        TF_dir_name = "/" + organism + "_" + model + "_" + str(size_of_set)


        # обертка для SARUS и построения матриц

        print(" ")
        #print("Selected_full_names_list:", len(selected_full_names_list)) # получили список факторов для человека и мыши
        сalc = 0
        for TF in sorted(selected_HUMAN): 
            print(TF)
            print("TF # {сalc} of {all_c}".format(all_c=len(selected_MOUSE), сalc=сalc)) # сколько факторов и на каком мы сейчас
            сalc += 1

            print("Dirname is:", TF) # делаем директорию для фактора

            new_dir_name = outputdir + "/"+ TF.split("_")[0] + "/" + seq_org + "_seq_" + pwm_org + "_pwm_di"
            #print("New_dir_name", new_dir_name)
            #print(" ")

            os.chdir(new_dir_name) # заходим в директорию фактора
            dir_list = os.listdir(pwmdir_di + "/" + TF)  # выбираем матицы для него же

            #print("dir_list", len(dir_list))
            #print(dir_list[:5])

            dir_list_short = [x.split("~")[2] for x in dir_list] # делаем короткие имена для поиска

            try:
                pwm_benchmark = pd.read_csv("~/hocomoco11/curation/slices4bench_di/out_sure.csv", sep='\t', header=None) # читаем таблицу со скорами для нашего фактора
                wlogauc_datasets = pd.read_csv("/home/pavelkrav/hocomoco11/wlogauc/di/" + TF.split("_")[1]+ "/out_sure.csv", sep='\t', header=None) # читаем таблицу со скорами для нашего фактора
                wlogauc_datasets["names"] = [ x.split("~")[2].split(".")[0] for x in wlogauc_datasets[0]]
                wlogauc_datasets = wlogauc_datasets[wlogauc_datasets["names"] == TF]
                wlogauc_datasets = wlogauc_datasets.sort_values(1)
    
                pwm_benchmark["names"] = [ x.split(".")[0] for x in pwm_benchmark[0]]
                pwm_benchmark = pwm_benchmark[pwm_benchmark["names"] == TF]
                pwm_benchmark = pwm_benchmark[["macs" in x for x in pwm_benchmark[0].str.split(".")]]
                pwm_benchmark = [x+".dpwm" for x in list(pwm_benchmark[0])]
                wlogauc_datasets = list(wlogauc_datasets[0])
                                 
                pwm_final_list = []
                
                for m in pwm_benchmark: # для полученных имен
                    if m in dir_list_short: # если их не было в директории при проверке, то удаляем из списка
                        pwm_final_list.append(dir_list[dir_list_short.index(m)])
                        
                #print("Before: ", len(pwm_final_list), pwm_final_list) # в итоге получили список отобранных

            except FileNotFoundError: # если нет описания для фактора, то просто берем все 
                print("No file !!!!!")
                continue

            matrix_c = 0 # начинаем считать матрицы 

            if len(pwm_final_list) < 3:
                continue

            for matrix in range(len(pwm_final_list)): # берем матрицу                    
                matrix = pwm_final_list[matrix]
                #print("PWM in process is:", matrix)
                matrix_dump = matrix

                if matrix_c < N_features: # если число матриц больше заданного
                    #print("Sarus is running ...") # запускаем генерацию фичей
                    #print(" ")
                    #print(matrix_dump)
                    #print(matrix_dump.split("~")[-1])
                    #print(matrix_dump.split("~")[-1][:-3] + "pcm")
                    matrix_pcm = matrix_dump.split("~")[-1][:-3] + "pcm"
                    matrix_pcm_out = new_dir_name+"/"+matrix_pcm
                    #print(matrix_pcm,matrix_pcm_out)

                    matrix_pcm_out2 = new_dir_name+"/"+matrix_dump.split("~")[-1][:-4] + "_2.dpcm"
                    matrix_png_out = new_dir_name+"/"+matrix_dump.split("~")[-1][:-4] + "png"
                    line = f"tail -n +2 {matrix_pcm_out} > {matrix_pcm_out2}; ruby /home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb {matrix_pcm_out2} {matrix_png_out}"
                    #print(line)
                    p = subprocess.Popen(line, shell=True)
                    p.wait()
                    
                else:
                    break
                matrix_c += 1

            os.chdir(outputdir) # выходим в директорию эксперимента


LLIB 04 Jun 01:10:04 [ytilib]	ytilib required, working directory /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di
LLIB 04 Jun 01:10:04 [ytilib]	pmflogo3.rb started, usage: <in_file_name.dpcm/fa/mfa> <out_file_name> [<x_unit>=100] [<y_unit>=200] [<mono_scheme>=nucl_simpa] [<di_scheme>=di_simpa]
LLIB 04 Jun 01:10:04 [ytilib]	/home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di/RXRA_HUMAN.PEAKS033436.macs.M._2.dpcm /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di/RXRA_HUMAN.PEAKS033436.macs.M.png
LLIB 04 Jun 01:10:04 [dpmflogo3_mod]	pseudo-D-KDIC = 15.870039512544679
LLIB 04 Jun 01:10:04 [dpmflogo3_mod]	weight = 500.0000000000001


[DEPRECATION] requiring "RMagick" is deprecated. Use "rmagick" instead


LLIB 04 Jun 01:10:05 [ytilib]	ytilib required, working directory /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di
LLIB 04 Jun 01:10:05 [ytilib]	pmflogo3.rb started, usage: <in_file_name.dpcm/fa/mfa> <out_file_name> [<x_unit>=100] [<y_unit>=200] [<mono_scheme>=nucl_simpa] [<di_scheme>=di_simpa]
LLIB 04 Jun 01:10:05 [ytilib]	/home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di/RXRA_HUMAN.PEAKS033797.macs.M._2.dpcm /home/pavelkrav/Release/TF-ML/outputdir/RXRA/HUMAN_seq_HUMAN_pwm_di/RXRA_HUMAN.PEAKS033797.macs.M.png
LLIB 04 Jun 01:10:05 [dpmflogo3_mod]	pseudo-D-KDIC = 15.926336278496422
LLIB 04 Jun 01:10:05 [dpmflogo3_mod]	weight = 500.0000000000001
SOX2_HUMAN
TF # 28 of 38
Dirname is: SOX2_HUMAN


[DEPRECATION] requiring "RMagick" is deprecated. Use "rmagick" instead


LLIB 04 Jun 01:10:06 [ytilib]	ytilib required, working directory /home/pavelkrav/Release/TF-ML/outputdir/SOX2/HUMAN_seq_HUMAN_pwm_di
LLIB 04 Jun 01:10:06 [ytilib]	pmflogo3.rb started, usage: <in_file_name.dpcm/fa/mfa> <out_file_name> [<x_unit>=100] [<y_unit>=200] [<mono_scheme>=nucl_simpa] [<di_scheme>=di_simpa]
LLIB 04 Jun 01:10:06 [ytilib]	/home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb /home/pavelkrav/Release/TF-ML/outputdir/SOX2/HUMAN_seq_HUMAN_pwm_di/SOX2_HUMAN.PEAKS033277.macs.M._2.dpcm /home/pavelkrav/Release/TF-ML/outputdir/SOX2/HUMAN_seq_HUMAN_pwm_di/SOX2_HUMAN.PEAKS033277.macs.M.png
LLIB 04 Jun 01:10:06 [dpmflogo3_mod]	pseudo-D-KDIC = 19.224407692520682
LLIB 04 Jun 01:10:06 [dpmflogo3_mod]	weight = 500.0000000000001


...

[DEPRECATION] requiring "RMagick" is deprecated. Use "rmagick" instead


LLIB 04 Jun 01:10:29 [ytilib]	ytilib required, working directory /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di
LLIB 04 Jun 01:10:29 [ytilib]	pmflogo3.rb started, usage: <in_file_name.dpcm/fa/mfa> <out_file_name> [<x_unit>=100] [<y_unit>=200] [<mono_scheme>=nucl_simpa] [<di_scheme>=di_simpa]
LLIB 04 Jun 01:10:29 [ytilib]	/home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di/SPI1_HUMAN.PEAKS035173.macs.M._2.dpcm /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di/SPI1_HUMAN.PEAKS035173.macs.M.png
LLIB 04 Jun 01:10:29 [dpmflogo3_mod]	pseudo-D-KDIC = 23.756067676506778
LLIB 04 Jun 01:10:29 [dpmflogo3_mod]	weight = 474.0000000000001


[DEPRECATION] requiring "RMagick" is deprecated. Use "rmagick" instead


LLIB 04 Jun 01:10:31 [ytilib]	ytilib required, working directory /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di
LLIB 04 Jun 01:10:31 [ytilib]	pmflogo3.rb started, usage: <in_file_name.dpcm/fa/mfa> <out_file_name> [<x_unit>=100] [<y_unit>=200] [<mono_scheme>=nucl_simpa] [<di_scheme>=di_simpa]
LLIB 04 Jun 01:10:31 [ytilib]	/home/ivavlakul/bin/chipmunk/dpmflogo3_mod.rb /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di/SPI1_HUMAN.PEAKS035176.macs.M._2.dpcm /home/pavelkrav/Release/TF-ML/outputdir/SPI1/HUMAN_seq_HUMAN_pwm_di/SPI1_HUMAN.PEAKS035176.macs.M.png
LLIB 04 Jun 01:10:31 [dpmflogo3_mod]	pseudo-D-KDIC = 23.361356699001167
LLIB 04 Jun 01:10:31 [dpmflogo3_mod]	weight = 496.0000000000001


[DEPRECATION] requiring "RMagick" is deprecated. Use "rmagick" instead
